# Notebook 12c — Caudal IDEAM en R (réplica con tidyverse del análisis Python del notebook 12b)

Replica en R, mediante el ecosistema `tidyverse`, el análisis de correlación entre el caudal medio mensual del río Magdalena en El Banco y del río Aracataca en Ganadería Caribe ---ambas series IDEAM/DHIME 2013--2025--- y las anomalías NDVI del manglar de la CGSM, ya desarrollado en Python en el notebook `12b_caudal_ideam_elbanco.ipynb`. La elección de R para este análisis específico obedece a que las series temporales hidrológicas se trabajan tradicionalmente en R en la literatura ecohidrológica, en este sentido `dplyr` + `lubridate` + `ggplot2` ofrecen una sintaxis más concisa y profesional para este tipo de análisis que pandas+matplotlib. La convergencia numérica entre los dos notebooks ---Python y R--- constituye una validación adicional del análisis y demuestra que las conclusiones del informe no dependen de la implementación.

**Insumos:** mismos que el notebook 12b.

**Productos:**
- `outputs/tables/caudal_ideam_correlacion_R.csv` (debe coincidir con `correlacion_caudal_ndvi.csv` salvo redondeo)
- `outputs/figures/caudal_vs_ndvi_correlacion_R.png`

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(readr)
  library(lubridate)
  library(ggplot2)
  library(purrr)
})

setwd('/home/rstudio/work/proyecto-cgsm')
BANCO    <- 'data/raw/ideam/descargaDhime_elbanco_medio.csv'
CARIBE   <- 'data/raw/ideam/descargaDhime_ganaderia_caribe.csv'
NDVI_CSV <- 'outputs/tables/serie_temporal_ndvi_definitiva.csv'
OUT_TAB  <- 'outputs/tables/caudal_ideam_correlacion_R.csv'
OUT_FIG  <- 'outputs/figures/caudal_vs_ndvi_correlacion_R.png'
cat('Setup R OK\n')

## 1. Carga y anomalías z-score de las dos series de caudal

In [ ]:
cargar_anomalia <- function(csv, etiqueta) {
  read_csv(csv, show_col_types = FALSE) %>%
    mutate(date = as.Date(Fecha) + days(14),
           caudal = as.numeric(Valor),
           estacion = etiqueta) %>%
    filter(!is.na(caudal)) %>%
    group_by(mes = month(date)) %>%
    mutate(caudal_z = (caudal - mean(caudal)) / sd(caudal)) %>%
    ungroup() %>%
    select(date, estacion, caudal, caudal_z)
}

caudal <- bind_rows(
  cargar_anomalia(BANCO,  'El_Banco_Magdalena'),
  cargar_anomalia(CARIBE, 'Ganaderia_Caribe_Aracataca')
)

caudal %>% group_by(estacion) %>%
  summarise(n = n(), media = mean(caudal), sd_z = sd(caudal_z)) %>%
  print()

## 2. Carga y desagregación del NDVI por naturaleza espectral

In [ ]:
manglar     <- c('Cano_Palos', 'Cano_Clarin', 'CP_Aguas_Negras', 'CP_Luna')
limnologica <- c('Isla_Boqueron', 'Punta_Cerro', 'Punta_Chino', 'Rio_Sevilla')

ndvi <- read_csv(NDVI_CSV, show_col_types = FALSE) %>%
  mutate(date = as.Date(date),
         naturaleza = case_when(
           subzona %in% manglar     ~ 'manglar',
           subzona %in% limnologica ~ 'limnologica',
           TRUE                     ~ 'otra'
         )) %>%
  group_by(subzona) %>%
  mutate(z = (ndvi - mean(ndvi, na.rm = TRUE)) / sd(ndvi, na.rm = TRUE)) %>%
  ungroup() %>%
  filter(naturaleza %in% c('manglar', 'limnologica'))

z_mensual <- ndvi %>%
  mutate(date = floor_date(date, 'month') + days(14)) %>%
  group_by(date, naturaleza) %>%
  summarise(z = mean(z, na.rm = TRUE), .groups = 'drop')

cat('NDVI mensual por naturaleza:\n')
z_mensual %>% group_by(naturaleza) %>% summarise(n = n()) %>% print()

## 3. Correlación caudal vs NDVI por estación, naturaleza y rezago (0--3 meses)

In [ ]:
estaciones  <- c('El_Banco_Magdalena', 'Ganaderia_Caribe_Aracataca')
naturalezas <- c('manglar', 'limnologica')
rezagos     <- 0:3

calcular_rho <- function(est, nat, lag) {
  serie_q <- caudal %>% filter(estacion == est) %>% select(date, caudal_z)
  serie_n <- z_mensual %>% filter(naturaleza == nat) %>% select(date, z)
  merged  <- inner_join(serie_n, serie_q, by = 'date') %>%
    arrange(date) %>%
    mutate(caudal_lag = lag(caudal_z, lag)) %>%
    drop_na()
  tibble(
    estacion     = est,
    naturaleza   = nat,
    rezago_meses = lag,
    rho_caudal   = round(cor(merged$z, merged$caudal_lag), 3),
    n            = nrow(merged)
  )
}

df_corr <- cross_join(
    tibble(estacion = estaciones),
    cross_join(tibble(naturaleza = naturalezas), tibble(rezago = rezagos))
  ) %>%
  pmap_dfr(~ calcular_rho(..1, ..2, ..3))

write_csv(df_corr, OUT_TAB)
print(df_corr, n = 16)
cat(sprintf('\nGuardado: %s\n', OUT_TAB))

## 4. Figura comparativa de correlaciones (ggplot2)

In [ ]:
p <- df_corr %>%
  mutate(
    estacion = factor(estacion,
      levels = c('El_Banco_Magdalena', 'Ganaderia_Caribe_Aracataca'),
      labels = c('El Banco — Magdalena central',
                 'Ganadería Caribe — Sierra Nevada (Aracataca)'))
  ) %>%
  ggplot(aes(x = factor(rezago_meses), y = rho_caudal, fill = naturaleza)) +
  geom_col(position = position_dodge(0.7), width = 0.6, alpha = 0.85) +
  geom_hline(yintercept = 0, color = 'black', linewidth = 0.4) +
  facet_wrap(~ estacion) +
  scale_fill_manual(values = c(manglar = '#2E7D32', limnologica = '#0277BD'),
                    name = 'Naturaleza') +
  labs(x = 'Rezago (meses)',
       y = expression(rho * ' Pearson (caudal medio vs NDVI z-score)'),
       title = 'Correlación caudal medio mensual IDEAM vs NDVI manglar CGSM',
       subtitle = '2013–2025 · Réplica en R (tidyverse) del análisis Python del notebook 12b') +
  theme_minimal(base_size = 11) +
  theme(plot.title = element_text(face = 'bold'))

ggsave(OUT_FIG, p, width = 11, height = 5, dpi = 200, bg = 'white')
print(p)
cat(sprintf('Guardado: %s\n', OUT_FIG))

## 5. Verificación de convergencia con la versión Python

Si los valores de $\rho$ calculados aquí coinciden ---salvo redondeo a tres decimales--- con los de la tabla `correlacion_caudal_ndvi.csv` producida por el notebook 12b, queda demostrado que el resultado del análisis de correlación caudal-NDVI no depende de la implementación ni del lenguaje, en este sentido la conclusión del informe ---correlaciones positivas universales que reflejan el efecto beneficioso del aporte fluvial sostenido sobre el manglar--- es robusta a la elección del entorno computacional.